# N165 · 信号量、条件变量与交替执行

**目标：** 维护轮到谁的状态并正确等待谓词。

**先修：** N164, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Semaphore；Condition；while predicate；notify；零奇偶状态机；忙等反例。

## 从问题到算法

条件变量的通知不是“轮到我”的证明。线程被唤醒后必须在同一把锁下用 while 重新检查 turn；否则错误唤醒或竞争可能破坏交替顺序。

FooBar 用状态机 0→1→0；ZeroEvenOdd 用三组许可证：初始只允许 zero，zero 根据下一数字的奇偶把唯一的工作许可转交给对应线程；数字线程打印后再把许可交回 zero。

每个打印回调都假定有限完成且不抛异常。把回调作为可失败的业务任务时，需额外设计取消与故障传播，不应悄悄把这类契约塞进教学算法。

## 最小实现

**本章接口：** `FooBar`、`ZeroEvenOdd`

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class FooBar:
    def __init__(self,n):
        if n < 0:
            raise ValueError('n must be nonnegative')
        self.n = n
        self.turn = 0
        self.condition = threading.Condition()
    def foo(self, printFoo):
        self._repeat(0,printFoo)
    def bar(self, printBar):
        self._repeat(1,printBar)
    def _repeat(self, turn, callback):
        for _ in range(self.n):
            with self.condition:
                while self.turn != turn:
                    self.condition.wait()
                callback()
                self.turn = 1-turn
                self.condition.notify_all()

class ZeroEvenOdd:
    def __init__(self,n):
        if n < 0:
            raise ValueError('n must be nonnegative')
        self.n = n
        self.zero_gate = threading.Semaphore(1)
        self.odd_gate = threading.Semaphore(0)
        self.even_gate = threading.Semaphore(0)
    def zero(self, printNumber):
        for value in range(1,self.n+1):
            self.zero_gate.acquire()
            printNumber(0)
            (self.odd_gate if value%2 else self.even_gate).release()
    def odd(self, printNumber):
        for value in range(1,self.n+1,2):
            self.odd_gate.acquire()
            printNumber(value)
            self.zero_gate.release()
    def even(self, printNumber):
        for value in range(2,self.n+1,2):
            self.even_gate.acquire()
            printNumber(value)
            self.zero_gate.release()
'''

import json
import subprocess
import sys

# This defines the visible APIs only; it does not launch a thread.
exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra],
        text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

FooBar 在锁内检查、打印、切换状态，整个转换是原子的。零奇偶算法每一步只开放下一类合法操作的许可，按 i 归纳得到 0,1,0,2,…,0,n。有限测试不证明调度公平；活性以相关线程最终被调度为前提。

**代价：** 输出 2n 个标记，算法状态 O(1)。记录测试轨迹需 O(n) 额外空间，但不是同步算法自身所需。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

rows = json.loads(run_isolated("""import json
obj=ZeroEvenOdd(5);trace=[]
run_threads([lambda:obj.even(trace.append),lambda:obj.odd(trace.append),lambda:obj.zero(trace.append)])
print(json.dumps([[i,value,'zero -> number' if value==0 else 'number -> zero'] for i,value in enumerate(trace)]))
"""))
show_table(['step','printed','permission transfer'],rows)

step,printed,permission transfer
0,0,zero -> number
1,1,number -> zero
2,0,zero -> number
3,2,number -> zero
4,0,zero -> number
5,3,number -> zero
6,0,zero -> number
7,4,number -> zero
8,0,zero -> number
9,5,number -> zero


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_isolated("""import itertools
for n in [0,1,3,10]:
    for order in itertools.permutations(range(2)):
        obj=FooBar(n);out=[]
        methods=[lambda:obj.foo(lambda:out.append('foo')),lambda:obj.bar(lambda:out.append('bar'))]
        run_threads([methods[i] for i in order])
        assert ''.join(out)=='foobar'*n
    for order in itertools.permutations(range(3)):
        obj=ZeroEvenOdd(n);out=[]
        methods=[lambda:obj.zero(out.append),lambda:obj.even(out.append),lambda:obj.odd(out.append)]
        run_threads([methods[i] for i in order])
        assert out==[x for i in range(1,n+1) for x in (0,i)]
assert threading.active_count()==1
print('Alternation, counts, and worker cleanup passed.')
"""))
print('N165: 所有本章断言通过')

Alternation, counts, and worker cleanup passed.

N165: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 分别以信号量和Condition实现交替。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释等待谓词为何使用while。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1115. Print FooBar Alternately（canonical）
- 1116. Print Zero Even Odd（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。